# COSC2753 Assignment 2 — Fashion Intelligence System

**Group \<number\>**

| Student ID | Name |
|---|---|
| s000000 | |
| s000000 | |
| s000000 | |
| s000000 | |

---

This notebook is the **assembled submission**. Each member develops in their own
notebook under `notebooks/`, and the final versions are pasted into the matching
section here at the end.

Setup instructions are in `README.md`. It must run top-to-bottom on a clean machine.

## 0. Setup

Puts the repo root on the import path so `from src...` resolves no matter where
Jupyter was launched from.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import SEED, TARGETS, OUTPUT_DIR, MODEL_DIR
from src.data import get_split, get_images_only, load_test_images, load_metadata

np.random.seed(SEED)
sns.set_theme(style="whitegrid")

print(f"repo root: {ROOT}")

## 1. Data loading and cleaning

All cleaning lives in `src/data.py` so every task trains on identical rows.
Steps applied: drop the two junk `Unnamed` columns, inner-join the CSV against
images that exist on disk, convert the 343 grayscale files to RGB, resize the 17
images that are not 60×80, and assign a fixed train/val split stratified on
`articleType`.

Missing labels are filtered **per target** rather than globally, because the
targets differ (`season` 20, `usage` 72, `articleType` and `gender` 0).

In [ ]:
meta = load_metadata()
meta.head()

## 2. Exploratory data analysis

> **M3** (metadata) and **M4** (images) — paste your final figures here.

In [ ]:
for t in TARGETS:
    vc = meta[t].value_counts()
    print(f"{t:12} {vc.size:3d} classes | majority '{vc.index[0]}' {vc.iloc[0] / vc.sum():.3f} "
          f"| missing {meta[t].isna().sum()}")

## 3. Task 1 — Item type classification (`articleType`)

> **M1** — baselines, CNN architecture, hyper-parameter tuning, resolution study.
>
> **M2** — class imbalance (weighting, resampling, hierarchical grouping) and error analysis.

124 classes, majority baseline 0.176. 79 classes hold under 100 images, so the
structural macro-F1 ceiling is roughly 0.37 — a score near 0.4 is a good result,
not a broken model.

In [ ]:
# X_train, X_val, y_train, y_val, le = get_split("articleType")

## 4. Task 2 — Season classification (`season`)

> **M3**

4 classes, majority baseline 0.495. Hypothesis: the label reflects a
merchandising calendar more than the garment's appearance, so the ceiling is in
the label rather than the model. Show evidence, don't assert it.

In [ ]:
# X_train, X_val, y_train, y_val, le = get_split("season")

## 5. Task 3 — Gender and occasion classification (`gender`, `usage`)

> **M3**

`gender` 5 classes, baseline 0.541. `usage` 8 classes, baseline 0.769 with a
macro-F1 ceiling of 0.50 — report accuracy and macro-F1 side by side, because the
gap between them is the finding.

Also covers the multi-task model: one shared trunk, four heads, compared against
the four single-task models.

In [ ]:
# X_train, X_val, y_train, y_val, le = get_split("gender")
# X_train, X_val, y_train, y_val, le = get_split("usage")

## 6. Task 4 — Visual search

> **M4**

Top-K retrieval over ~38.6k images, built as a ladder so the comparison is real:
raw pixels → PCA → autoencoder → CNN penultimate → triplet loss. Evaluated with
Precision@{1,5,10} and mAP@10, plus a qualitative query→top-5 grid.

In [ ]:
# X, meta = get_images_only()   # unsupervised — no target filtering

## 7. Model comparison and ultimate judgement

> **Whole group**

Every model in one table, each quoted against its majority baseline. Then the
beyond-metrics analysis the spec requires for higher grades: calibration, error
analysis, learning curves, robustness, inference cost, and subgroup fairness.

## 8. Final predictions

> **M1**

Load the saved models and label encoders — **never refit an encoder here**, as
that silently reorders the classes and produces a wrong submission file.

Verify before submitting: 5,829 rows, ids matching `styles_prediction.csv` exactly
and in order, column names unchanged, no NaNs, every label drawn from the
training vocabulary.

In [ ]:
# X_test, submission = load_test_images()
# submission.to_csv(OUTPUT_DIR / "styles_prediction.csv", index=False)